# MLP — Multilayer Perceptron for Spike Classification
Binary classification of next-hour electricity price spikes using a flat feature vector.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_curve, auc

from shared.data_prep import (
    load_data, split_data, get_feature_cols, fit_scaler, apply_scaler,
    compute_pos_weight, random_search, TabularDataset,
    train_model, evaluate, TARGET
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
print(f"Device: {DEVICE}")

## 1. Data Loading and Feature Preparation

In [ ]:
df = load_data()
train, val, test = split_data(df)

feature_cols = get_feature_cols("MLP", df.columns.tolist())
print(f"Features: {len(feature_cols)}")
print(f"Train rows: {len(train):,} | Val rows: {len(val):,} | Test rows: {len(test):,}")

# Fit scaler on training data only
scaler, cont_cols = fit_scaler(train, feature_cols)

train_s = apply_scaler(train, scaler, cont_cols)
val_s   = apply_scaler(val,   scaler, cont_cols)
test_s  = apply_scaler(test,  scaler, cont_cols)

# Class imbalance weight (computed from training labels only)
pos_weight = compute_pos_weight(train[TARGET])
print(f"pos_weight: {pos_weight.item():.2f}  (n_neg/n_pos)")

In [ ]:
BATCH_SIZE = 512

train_ds = TabularDataset(train_s, feature_cols)
val_ds   = TabularDataset(val_s,   feature_cols)
test_ds  = TabularDataset(test_s,  feature_cols)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE * 2, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 2. Model Architecture

In [ ]:
class MLP(nn.Module):
    """
    Fully-connected network with BatchNorm, ReLU, and Dropout.
    Outputs a single logit for BCEWithLogitsLoss.
    """
    def __init__(self, input_dim: int, hidden_dims: tuple, dropout: float):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)

## 3. Hyperparameter Search
Random search with TimeSeriesSplit (5 folds) on the combined train+val period.
Primary optimisation metric: validation F1-score.

In [ ]:
INPUT_DIM = len(feature_cols)

PARAM_GRID = {
    "hidden_dims": [(256, 128), (256, 128, 64), (512, 256, 128), (512, 256, 128, 64)],
    "dropout":     [0.2, 0.3, 0.5],
    "lr":          [1e-3, 5e-4, 1e-4],
    "weight_decay":[1e-4, 1e-5, 0.0],
    "batch_size":  [256, 512, 1024],
}
N_TRIALS = 20

# Combined train+val for cross-validation
train_val = pd.concat([train, val], ignore_index=True)

def build_fn(params, fold_train_df):
    pw = compute_pos_weight(fold_train_df[TARGET])
    model     = MLP(INPUT_DIM, params["hidden_dims"], params["dropout"])
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=params["lr"],
                                 weight_decay=params["weight_decay"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw.to(DEVICE))
    return model, optimizer, criterion

search_results = random_search(
    PARAM_GRID, N_TRIALS, build_fn, train_val, feature_cols, DEVICE,
    use_sequences=False, n_cv_splits=5, max_epochs=30, patience=5
)

print("\n\u2500\u2500 Top 5 Configurations \u2500\u2500")
for r in search_results[:5]:
    print(f"  F1={r['mean_cv_f1']:.4f}  {r['params']}")

In [ ]:
best_params = search_results[0]["params"]
print("Best hyperparameters:", best_params)

## 4. Final Training
Train on the full training set with the best hyperparameters.
Early stopping on validation F1 (patience=10).

In [ ]:
# Re-scale using full training set (not fold-level)
final_scaler, final_cont_cols = fit_scaler(train, feature_cols)
train_sf  = apply_scaler(train, final_scaler, final_cont_cols)
val_sf    = apply_scaler(val,   final_scaler, final_cont_cols)
test_sf   = apply_scaler(test,  final_scaler, final_cont_cols)

final_pw = compute_pos_weight(train[TARGET])

bs = best_params["batch_size"]
train_ds_f = TabularDataset(train_sf, feature_cols)
val_ds_f   = TabularDataset(val_sf,   feature_cols)
test_ds_f  = TabularDataset(test_sf,  feature_cols)

train_loader_f = DataLoader(train_ds_f, batch_size=bs, shuffle=True, drop_last=True)
val_loader_f   = DataLoader(val_ds_f,   batch_size=bs * 2, shuffle=False)
test_loader_f  = DataLoader(test_ds_f,  batch_size=bs * 2, shuffle=False)

final_model = MLP(INPUT_DIM, best_params["hidden_dims"], best_params["dropout"]).to(DEVICE)
final_optimizer = torch.optim.Adam(final_model.parameters(),
                                   lr=best_params["lr"],
                                   weight_decay=best_params["weight_decay"])
final_criterion = nn.BCEWithLogitsLoss(pos_weight=final_pw.to(DEVICE))

history, best_val_f1 = train_model(
    final_model, train_loader_f, val_loader_f,
    final_optimizer, final_criterion, DEVICE,
    max_epochs=50, patience=10,
    checkpoint_dir=CHECKPOINT_DIR
)
print(f"\nBest validation F1: {best_val_f1:.4f}")

In [ ]:
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="Train loss")
axes[0].plot(hist_df["epoch"], hist_df["loss"],       label="Val loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(hist_df["epoch"], hist_df["f1"],  label="Val F1")
axes[1].plot(hist_df["epoch"], hist_df["auc"], label="Val AUC")
axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.suptitle("MLP Training History")
plt.tight_layout()
plt.show()

## 5. Evaluation
Load the best checkpoint and evaluate on validation and test sets.

In [ ]:
# Load best checkpoint
best_model = MLP(INPUT_DIM, best_params["hidden_dims"], best_params["dropout"]).to(DEVICE)
best_model.load_state_dict(torch.load(CHECKPOINT_DIR / "best_model.pt", map_location=DEVICE))

val_metrics  = evaluate(best_model, val_loader_f,  final_criterion, DEVICE)
test_metrics = evaluate(best_model, test_loader_f, final_criterion, DEVICE)

results_df = pd.DataFrame([val_metrics, test_metrics], index=["Validation", "Test"])
print(results_df.round(4))

In [ ]:
def get_preds_probs(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            all_logits.append(model(X).cpu())
            all_labels.append(y)
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)
    probs  = 1 / (1 + np.exp(-logits))
    preds  = (probs >= 0.5).astype(int)
    return labels, probs, preds

val_labels,  val_probs,  val_preds  = get_preds_probs(best_model, val_loader_f,  DEVICE)
test_labels, test_probs, test_preds = get_preds_probs(best_model, test_loader_f, DEVICE)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix — test set
ConfusionMatrixDisplay.from_predictions(test_labels, test_preds, ax=axes[0],
                                        display_labels=["No Spike", "Spike"],
                                        cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix \u2014 Test Set (MLP)")

# ROC curve — both sets
for labels, probs, name in [(val_labels, val_probs, "Validation"),
                             (test_labels, test_probs, "Test")]:
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[1].set_title("ROC Curve \u2014 MLP")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nTest Set Classification Report:")
print(classification_report(test_labels, test_preds, target_names=["No Spike", "Spike"]))

In [ ]:
# Gradient-based feature importance: mean |grad| per feature on test set
best_model.eval()
test_loader_single = DataLoader(test_ds_f, batch_size=512, shuffle=False)
grad_sum = torch.zeros(len(feature_cols))
n_samples = 0
for X, y in test_loader_single:
    X = X.to(DEVICE).requires_grad_(True)
    logits = best_model(X)
    logits.sum().backward()
    grad_sum += X.grad.abs().mean(0).cpu().detach()
    n_samples += 1

importance = (grad_sum / n_samples).numpy()
top_idx = np.argsort(importance)[::-1][:20]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh([feature_cols[i] for i in top_idx[::-1]],
        importance[top_idx[::-1]], color="steelblue")
ax.set_title("Top 20 Features by Mean |Gradient| \u2014 MLP")
ax.set_xlabel("Mean |Gradient|")
plt.tight_layout()
plt.show()

## 6. Summary

In [ ]:
print("=" * 50)
print("MLP FINAL RESULTS")
print("=" * 50)
print(f"Best hyperparameters : {best_params}")
print(f"\nValidation set:")
for k, v in val_metrics.items():
    print(f"  {k:12s}: {v:.4f}")
print(f"\nTest set (out-of-sample):")
for k, v in test_metrics.items():
    print(f"  {k:12s}: {v:.4f}")
print(f"\nCheckpoints saved to: {CHECKPOINT_DIR.resolve()}")